In [1]:
import pandas as pd
from scipy import stats

sales = pd.read_csv("E:\CODING_ALL\Data Analytics\projrcts\RetailIQ\data\Cleaned csv\cleaned_sales.csv")
items = pd.read_csv("E:\CODING_ALL\Data Analytics\projrcts\RetailIQ\data\Cleaned csv\cleaned_orders.csv")

# Calculate profit margin
items["profit_margin"] = (
    items["profit"] / items["net_sales"] * 100
)

# Calculate order-level revenue and profit margin
order_stats = items.groupby("order_id").agg(
    order_revenue=("net_sales", "sum"),
    order_profit=("profit", "sum")
).reset_index()

order_stats["profit_margin"] = (
    order_stats["order_profit"] /
    order_stats["order_revenue"] * 100
)

data = sales.merge(order_stats, on="order_id", how="left")

## Q-1 Are larger discounts associated with lower profit margins?

In [2]:
discount_data = items[
    ["discount_percentage", "profit_margin"]
].dropna()

correlation, p_value = stats.spearmanr(
    discount_data["discount_percentage"],
    discount_data["profit_margin"]
)

print("Spearman correlation:", round(correlation, 3))
print("P-value:", round(p_value, 4))

if p_value < 0.05:
    print("The relationship is statistically significant.")
else:
    print("No statistically significant relationship found.")

Spearman correlation: -0.514
P-value: 0.0
The relationship is statistically significant.


## Q-2 Do delayed orders have different customer ratings?

In [3]:
data["delay_days"] = (
    data["delivery_days"] - data["estimated_delivery_days"]
)

ratings = data.dropna(
    subset=["delay_days", "customer_rating"]
)

delayed = ratings.loc[
    ratings["delay_days"] > 0, "customer_rating"
]

not_delayed = ratings.loc[
    ratings["delay_days"] <= 0, "customer_rating"
]

if len(delayed) > 0 and len(not_delayed) > 0:
    statistic, p_value = stats.mannwhitneyu(
        delayed, not_delayed, alternative="two-sided"
    )

    print("Average rating — delayed:", round(delayed.mean(), 2))
    print("Average rating — not delayed:", round(not_delayed.mean(), 2))
    print("P-value:", round(p_value, 4))
else:
    print("Not enough data in both groups.")

Average rating — delayed: 3.22
Average rating — not delayed: 3.76
P-value: 0.0


## Q-3 Do customer segments have different average order values?

In [5]:
# Load customer data
customers = pd.read_csv("E:\CODING_ALL\Data Analytics\projrcts\RetailIQ\data\Cleaned csv\cleaned_customers.csv")

# Add customer segment to order data
data = data.merge(
    customers[["customer_id", "customer_segment"]],
    on="customer_id",
    how="left"
)

# Compare average order values by customer segment
segment_data = data.dropna(
    subset=["customer_segment", "order_revenue"]
)

groups = [
    group["order_revenue"].values
    for _, group in segment_data.groupby("customer_segment")
]

print(
    segment_data.groupby("customer_segment")["order_revenue"]
    .agg(["count", "mean", "median"])
)

if len(groups) >= 2:
    statistic, p_value = stats.kruskal(*groups)

    print("\nKruskal-Wallis statistic:", round(statistic, 3))
    print("P-value:", round(p_value, 4))

    if p_value < 0.05:
        print("At least one customer segment differs significantly.")
    else:
        print("No statistically significant difference found.")

                  count         mean    median
customer_segment                              
Business          13777  1402.544567  1018.930
Consumer          75636  1396.309861  1031.575
Premium           34746  1384.939078  1020.215
VIP               13957  1385.945043  1028.730

Kruskal-Wallis statistic: 1.947
P-value: 0.5834
No statistically significant difference found.
